# 04 – Modeller arası karşılaştırma (eşleşik)

01/02/03 notebook'ları aynı sabit ızgarayı ürettiği için tahminler (görüntü, hastalık) anahtarıyla birebir eşleşir.

In [ ]:
import sys, os, json; sys.path.insert(0, "src")
import numpy as np, pandas as pd
import cxr_data as D, cxr_eval as E

SMOKE = False
RES_DIR = "results_smoke" if SMOKE else "results"
N_BOOT = 20 if SMOKE else 500
sp = D.load_splits("splits")
names = ["blip", "vilt", "clip"]
preds = {n: pd.read_csv(f"{RES_DIR}/{n}_test_predictions.csv.gz") for n in names}
for n, p in preds.items(): print(n, p.shape)
# ızgaraların birebir aynı olduğunu doğrula
keys = [set(zip(p.image_path, p.disease)) for p in preds.values()]
assert keys[0] == keys[1] == keys[2], "Izgaralar farklı! Üç model de aynı splits/ ile çalıştırılmalı."

In [ ]:
# Ana tablo: her model için Full / Shuffle / Black / Text-prior
rows = []
for n, p in preds.items():
    p["prob_prior"] = E.text_prior_baseline(p, sp.train, sp.label_cols)
    h = E.headline_table(p, {"Full": "prob_orig", "Noise": "prob_noise", "Black": "prob_black", "Text prior": "prob_prior",
                             "Shuffle-32": "prob_shuf32", "Shuffle-16": "prob_shuf16", "Shuffle-8": "prob_shuf8"})
    h.insert(0, "model", n); rows.append(h)
T = pd.concat(rows, ignore_index=True); T.to_csv(f"{RES_DIR}/compare_headline.csv", index=False); display(T.round(4))

In [ ]:
# ΔAUC + hasta-küme bootstrap GA + eşleşik Wilcoxon (mutlak hata)
res = E.compare_models(preds, col="prob_orig", n_boot=N_BOOT)
for k, v in res.items():
    v.to_csv(f"{RES_DIR}/compare_{k}.csv", index=False); print(k); display(v.round(4))

In [ ]:
# Görsel bağımlılık (eşleşik ΔP) — her model için
vd = pd.concat([E.visual_dependence_table(p).assign(model=n) for n, p in preds.items()], ignore_index=True)
vd.to_csv(f"{RES_DIR}/compare_visual_dependence.csv", index=False); display(vd.round(4))

In [ ]:
# Shuffle altında Full'a göre düşüş (ΔAUC) ve %95 GA — model başına
out = []
for n, p in preds.items():
    b = E.bootstrap_auc(p, ["prob_orig", "prob_shuf32", "prob_shuf16", "prob_shuf8"],
                        diffs=[("prob_orig", "prob_shuf32"), ("prob_orig", "prob_shuf16"), ("prob_orig", "prob_shuf8")], n_boot=N_BOOT)
    out.append(b.assign(model=n))
B = pd.concat(out, ignore_index=True); B.to_csv(f"{RES_DIR}/compare_shuffle_bootstrap.csv", index=False); display(B.round(4))